# Flight Price Data Preprocessing for Machine Learning

Objective:

This notebook performs:
- Data loading
- Exploratory Data Analysis (EDA)
- Data Cleaning
- Feature Engineering
- Data Preprocessing for Machine Learning

# 1. Import Libraries

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Load Dataset

In [41]:
df= pd.read_excel('flight_price.xlsx')

# 3. Basic Data Exploration

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Date_of_Journey  10683 non-null  str  
 2   Source           10683 non-null  str  
 3   Destination      10683 non-null  str  
 4   Route            10682 non-null  str  
 5   Dep_Time         10683 non-null  str  
 6   Arrival_Time     10683 non-null  str  
 7   Duration         10683 non-null  str  
 8   Total_Stops      10682 non-null  str  
 9   Additional_Info  10683 non-null  str  
 10  Price            10683 non-null  int64
dtypes: int64(1), str(10)
memory usage: 918.2 KB


In [43]:
## 11 columns and 10683 rows (entries)
## 0 to 9 are str ,so we might need to convert them into int


In [44]:
df.describe()

,Price
count,10683.000000
mean,9087.064121
std,4611.359167
min,1759.000000
25%,5277.000000
50%,8372.000000
75%,12373.000000
max,79512.000000


In [ ]:
## Date of Journey is object - with FE ,convert it into day month and year
df['Date']= df['Date_of_Journey'].str.split('/').str[0]
df['Month']=df['Date_of_Journey'].str.split('/').str[1]
df['Year']=df['Date_of_Journey'].str.split('/').str[2]

In [47]:
##Converting Date ,Month ,Year (objeccts) into int
df['Date']=df['Date'].astype(int)
df['Month']=df['Month'].astype(int)
df['Year']=df['Year'].astype(int)

In [49]:
##Drop Date of Journey Column
df.drop('Date_of_Journey',axis=1,inplace=True)

In [ ]:
## Arrival time 
## first let's remove the date and other parts so that we are left with time
df['Arrival_Time']=df['Arrival_Time'].str.split(' ').str[0]

In [52]:
##Let's make hour and minute columns
df['Arrival_Hour']=df['Arrival_Time'].str.split(':').str[0]
df['Arrival_Minute']=df['Arrival_Time'].str.split(':').str[1]

In [53]:
df['Arrival_Hour']=df['Arrival_Hour'].astype(int)
df['Arrival_Minute']=df['Arrival_Minute'].astype(int)
df.drop('Arrival_Time',axis=1,inplace=True)

In [55]:
df['Dep_Hour']=df['Dep_Time'].str.split(':').str[0]
df['Dep_Minute']=df['Dep_Time'].str.split(':').str[1]


In [56]:
df.drop('Dep_Time',axis=1,inplace=True)

In [57]:
df['Dep_Hour']=df['Dep_Hour'].astype(int)
df['Dep_Minute']=df['Dep_Minute'].astype(int)

In [59]:
df['Total_Stops'].unique()

<StringArray>
['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops']
Length: 6, dtype: str

In [60]:
df['Total_Stops'].mode()

0    1 stop
Name: Total_Stops, dtype: str

In [61]:
df['Total_Stops']=df['Total_Stops'].map({'non-stop':0,'2 stops':2,'1 stop':1,'3 stops':3,'4 stops':4, np.nan :1})
##Mode Imputation

In [62]:
df[df['Total_Stops'].isnull()]

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dep_Hour,Dep_Minute


In [63]:
df.drop('Route',axis=1,inplace=True)

In [64]:
##Duration
df["Duration_Total_Minutes"] = (
    pd.to_timedelta(df["Duration"])
    .dt.total_seconds()
    / 60
).astype(int)

In [65]:
df[['Duration','Duration_Total_Minutes']]

,Duration,Duration_Total_Minutes
0,2h 50m,170
1,7h 25m,445
2,19h,1140
3,5h 25m,325
4,4h 45m,285
...,...,...
10678,2h 30m,150
10679,2h 35m,155
10680,3h,180
10681,2h 40m,160


In [66]:
df.drop('Duration',axis=1,inplace=True)

In [67]:
df["Additional_Info"].value_counts()

Additional_Info
No info                         8345
In-flight meal not included     1982
No check-in baggage included     320
1 Long layover                    19
Change airports                    7
Business class                     4
No Info                            3
1 Short layover                    1
Red-eye flight                     1
2 Long layover                     1
Name: count, dtype: int64

In [68]:
df.drop("Additional_Info", axis=1, inplace=True)

The Additional_Info column was removed because it contained mostly "No info" values and contributed very little useful information.

Airline, Source, and Destination are categorical variables. They are converted into numerical features using One-Hot Encoding.

In [69]:

from sklearn.preprocessing import OneHotEncoder

# Columns containing category/text values
columns_to_encode = ["Airline", "Source", "Destination"]

# Create encoder
encoder = OneHotEncoder()




In [70]:
# Convert text categories into 0/1 columns
encoded_array = encoder.fit_transform(
    df[columns_to_encode]
).toarray()


In [71]:
# Convert the result into a DataFrame
encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoder.get_feature_names_out(columns_to_encode),
    index=df.index
)

# Remove original text columns and attach encoded columns
df = pd.concat(
    [df.drop(columns=columns_to_encode), encoded_df],
    axis=1
)


# Final Dataset

The dataset has now been:

- Cleaned
- Feature engineered
- Encoded
- Prepared for machine learning models

In [72]:
df.head()

,Total_Stops,Price,Date,Month,Year,Arrival_Hour,Arrival_Minute,Dep_Hour,Dep_Minute,Duration_Total_Minutes,...,Source_Chennai,Source_Delhi,Source_Kolkata,Source_Mumbai,Destination_Banglore,Destination_Cochin,Destination_Delhi,Destination_Hyderabad,Destination_Kolkata,Destination_New Delhi
0,0,3897,24,3,2019,1,10,22,20,170,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,7662,1,5,2019,13,15,5,50,445,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,2,13882,9,6,2019,4,25,9,25,1140,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1,6218,12,5,2019,23,30,18,5,325,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,1,13302,1,3,2019,21,35,16,50,285,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 33 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Total_Stops                                10683 non-null  int64  
 1   Price                                      10683 non-null  int64  
 2   Date                                       10683 non-null  int64  
 3   Month                                      10683 non-null  int64  
 4   Year                                       10683 non-null  int64  
 5   Arrival_Hour                               10683 non-null  int64  
 6   Arrival_Minute                             10683 non-null  int64  
 7   Dep_Hour                                   10683 non-null  int64  
 8   Dep_Minute                                 10683 non-null  int64  
 9   Duration_Total_Minutes                     10683 non-null  int64  
 10  Airline_Air Asia                 

## Conclusion

The dataset has been successfully preprocessed by:

- Extracting date features
- Extracting departure and arrival time features
- Converting duration into total minutes
- Encoding categorical variables using One-Hot Encoding
- Removing unnecessary columns

The processed dataset is now ready for model training.

In [74]:
# Save the processed dataset
df.to_csv("flight_price_processed.csv", index=False)